In [1]:
# Test data repair scripts for NY

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))

from calabasas_data_repair import data_repair
MY_JURISDICTION = "Calabasas"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_la_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 99.7% -> 99.8%
PERMIT_DATE available (all): 89.0% -> 89.3%
FINAL_DATE available (all): 66.5% -> 66.8%
PERMIT_DATE available (active/final): 94.3% -> 95.1%
FINAL_DATE available (final): 88.7% -> 89.6%


In [4]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Final        1369
Inactive      134
Active         53
In Review      48
Name: count, dtype: int64
STATUS_NORMALIZED
Final        1485
Active        184
In Review     164
Inactive      163
Name: count, dtype: int64


In [6]:
mask = sub_df_filled["FINAL_DATE"].isna() & (sub_df_filled["STATUS_NORMALIZED"] == "Final")
#mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Final    *Filled: nan*
RECORD_TYPE_ORIGINAL: Planning - Zoning clearance
FILE_DATE: 2006-09-15       *Filled: nan*
PERMIT_DATE: None   *Filled: nan*
FINAL_DATE: None     *Filled: nan*
DATES_DATA: 
{
  "My Project": {
    "Submitted": "9/15/2006"
  },
  "Build Status": "Permit Finaled"
}


In [7]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "Department": "Planning",
  "My Project": {
    "Closed": " - -",
    "Issued": " - -",
    "Street": "23353 PARK COLOMBO",
    "Created": " - -",
    "Approved": " - -",
    "Submitted": "9/15/2006",
    "Parcel Number": "2069-067-004",
    "CityStateZipcode": "Calabasas, CA 91302"
  },
  "Permit Fees": [],
  "Permit Type": "Zoning clearance",
  "Build Status": "Permit Finaled",
  "Parcel Number": "2069-067-004",
  "Permit Number": "ZC6000630",
  "Permit Details": {},
  "Permit Parcels": [],
  "Permit Contacts": [
    {
      "contact-name": [
        "LEGACY UNKNOWN"
      ],
      "contact-role": [
        "Submitter of the Application"
      ],
      "contact-notes": [],
      "contact-detail": [
        "Email not on record",
        "Phone number not on record"
      ]
    },
    {
      "contact-name": [
        "ALAN AND JUDY TRS NUSSEN NUSSENBLATT"
      ],
      "contact-role": [
        "Owner of Record - Parcel"
      ],
      "contact-notes": [],
      "contact-d